In [ ]:
USE {
	dependencies("org.duckdb:duckdb_jdbc:1.5.5.1")
}

In [ ]:
%use dataframe
%use lets-plot
%use kandy
%use plotly

import kotlinx.serialization.json.Json

In [ ]:
import org.jetbrains.kotlinx.dataframe.io.DbConnectionConfig
import org.jetbrains.kotlinx.dataframe.schema.DataFrameSchema

In [ ]:
val url = "jdbc:duckdb:/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_time_control_results_2026-08-27/time_control.duckdb"
val username = "duckdb"
val password = "password"

val dbConfig = DbConnectionConfig(url, username, password)

val tableName = "search_metrics"

val df = DataFrame.readSqlTable(dbConfig, tableName)



In [ ]:
df.head()

In [ ]:
df.schema()

In [ ]:
@DataSchema
interface Branch {
	@ColumnName("avg_branching_factor")
	val avgBranchingFactor: Double
	val depth: Double
	@ColumnName("eff_branching_factor")
	val effBranchingFactor: Double
	val model: String
	val seed: String
	val turn: Double
}

In [ ]:
val dir =
    "/Users/darronporter/Downloads/Dissertation/code/output/results/beelink_2026-08-19-search-data_run-1"

val branchingFactor =
    DataFrame.readCsv(
        "$dir/agent-1_MCTS_EASY_agent-2_MCTS_EASY_search_statistics.csv",
        header =
            listOf(
                "seed",
                "turn",
                "model",
                "depth",
                "avg_branching_factor",
                "eff_branching_factor",
            ),
	    parserOptions = ParserOptions(nullStrings = setOf("not assigned")),
    ).drop(1)

In [ ]:
branchingFactor.columnTypes()

In [ ]:
val stats = df

In [ ]:
stats.describe {
	turn and avg_branching_factor and eff_branching_factor
}

In [ ]:
val margin = 1.96 * stats.avg_branching_factor.std() / sqrt(stats.avg_branching_factor.size().toDouble())
margin

In [ ]:
// 1. Group by turn and compute the mean branching factor
val turnMeans = stats
	.select { "turn" and "avg_branching_factor" }
	.groupBy("turn")
	.mean("avg_branching_factor")

// 2. Plot density curve using Lets-Plot
val plot = letsPlot(turnMeans.toMap()) { x = "avg_branching_factor" } +
		geomDensity(
			fill = "#3B82F6",
			color = "#1D4ED8",
			alpha = 0.5,
			size = 1.0
		) +
		ggtitle("Density Distribution of Mean Branching Factor per Turn") +
		xlab("Mean Average Branching Factor") +
		ylab("Density")

plot

In [ ]:
// 1. Group by turn and compute the mean branching factor
val turnMeans = stats
	.select { "turn" and "avg_branching_factor" }
	.groupBy("turn")
	.mean("avg_branching_factor")

// 2. Plot density curve using Lets-Plot
val plot = letsPlot(turnMeans.toMap()) { x = "avg_branching_factor" } +
		geomHistogram(
			fill = "#3B82F6",
			color = "#1D4ED8",
			alpha = 0.5,
			size = 1.0
		) +
		ggtitle("Density Distribution of Mean Branching Factor per Turn") +
		xlab("Mean Average Branching Factor") +
		ylab("Density")

plot

In [ ]:
val branchingStats = stats
	.groupBy("turn")
	.aggregate {
		mean("avg_branching_factor") into "mean_branching"
		min("avg_branching_factor") into "min_branching"
		max("avg_branching_factor") into "max_branching"
	}

In [ ]:
val branchingMinMaxStats = stats
	.groupBy("turn")
	.aggregate {
		min("avg_branching_factor") into "min_branching"
		max("avg_branching_factor") into "max_branching"
	}

In [ ]:
val branchingMeanStats = stats
	.groupBy("turn")
	.aggregate {
		mean("avg_branching_factor") into "mean_branching"
	}

In [ ]:
branchingMeanStats

In [ ]:
val ribbonPlot = letsPlot { x = "turn"; y="mean_branching"; ymin="min_branching" ; ymax="max_branching" } +
		// Shaded area representing min-max range
		geomRibbon(
			data = branchingMinMaxStats.toMap(),
			fill = "#93C5FD",
			alpha = 0.4
		) +
		// Mean line
		geomLine(data = branchingMeanStats.toMap(), color = "#1D4ED8", size = 1.2) +
		geomPoint(data = branchingMeanStats.toMap(), color = "#1D4ED8", size = 2.5) +
		ggtitle("Branching Factor Progression Across Turns (Mean & Range)") +
		xlab("Turn") +
		ylab("Average Branching Factor")

ribbonPlot

In [ ]:
// 1. Calculate mean and standard deviation per turn
val turnBranchingStats = stats
	.groupBy("turn")
	.aggregate {
		mean("avg_branching_factor") into "mean"
		std("avg_branching_factor") into "std"
	}
	// 2. Derive ±1 and ±2 standard deviation boundaries (clamping at 0 if needed)
	.add("sd1_lower") { max(0.0, "mean"<Double>() - "std"<Double>()) }
	.add("sd1_upper") { "mean"<Double>() + "std"<Double>() }
	.add("sd2_lower") { max(0.0, "mean"<Double>() - 2 * "std"<Double>()) }
	.add("sd2_upper") { "mean"<Double>() + 2 * "std"<Double>() }
	.sortBy("turn")

In [ ]:
val plotWithStdDev =
    letsPlot(turnBranchingStats.toMap()) {
      x = "turn"
      y = "mean"
      ymin = "sd1_lower"
      ymax = "sd1_upper"
    } +
        // ±2 Standard Deviations (Outer Band ~95%)
        geomRibbon(
	        turnBranchingStats.select {"turn" and "sd2_lower" and "sd2_upper" }.toMap(),
            fill = "#93C5FD", // Light Blue
            alpha = 0.25,
        ) +
        // ±1 Standard Deviation (Inner Band ~68%)
        geomRibbon(
	        turnBranchingStats.select {"turn" and "sd1_lower" and "sd1_upper" }.toMap(),
            fill = "#3B82F6", // Medium Blue
            alpha = 0.35,
        ) +
        // Mean Trajectory Line & Points
        geomLine(
	        turnBranchingStats.select {"turn" and "mean" }.toMap(),
            color = "#1D4ED8", // Dark Blue
            size = 1.3,
        ) +
        geomPoint(
	        turnBranchingStats.select {"turn" and "mean" }.toMap(),
            color = "#1D4ED8",
            size = 2.5,
        ) +
        ggtitle("Average Branching Factor vs. Turn (with ±1σ Bands)") +
        xlab("Turn Number") +
        ylab("Average Branching Factor")

plotWithStdDev


In [ ]:
val plotWithStdDev =
	letsPlot(turnBranchingStats.toMap()) {
		x = "turn"
		y = "mean"
		ymin = "sd2_lower"
		ymax = "sd2_upper"
	} +
			// ±2 Standard Deviations (Outer Band ~95%)
			geomRibbon(
				turnBranchingStats.select {"turn" and "sd2_lower" and "sd2_upper" }.toMap(),
				fill = "#93C5FD", // Light Blue
				alpha = 0.25,
			) +
			// Mean Trajectory Line & Points
			geomLine(
				turnBranchingStats.select {"turn" and "mean" }.toMap(),
				color = "#1D4ED8", // Dark Blue
				size = 1.3,
			) +
			geomPoint(
				turnBranchingStats.select {"turn" and "mean" }.toMap(),
				color = "#1D4ED8",
				size = 2.5,
			) +
			ggtitle("Average Branching Factor vs. Turn (with ±2σ Bands)") +
			xlab("Turn Number") +
			ylab("Average Branching Factor")

plotWithStdDev
